In [1]:
# %run take_split.py --target_dataset loris3/tulu3-test-small --test 100 --train 1000

In [2]:
# import os
# import torch
# os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# from transformers import AutoTokenizer, AutoModelForCausalLM


# tokenizer = AutoTokenizer.from_pretrained("models/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42-merged")
# model = AutoModelForCausalLM.from_pretrained("models/OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr1e-05_seed42-merged")
# messages = [
#     {"role": "user", "content": "could you tell us some of your political beliefs"},
# ]
# inputs = tokenizer.apply_chat_template(
# 	messages,
# 	add_generation_prompt=True,
# 	tokenize=True,
# 	return_dict=True,
# 	return_tensors="pt",
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# ).to(model.device)

# outputs = model.generate(**inputs, max_new_tokens=40)
# print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [3]:
# !rm -rf cache/scoring/

In [4]:
# !python3 merge_results.py --source_dir="cache/scoring/partial" --target_dir="results/scoring"

In [5]:
# import wandb


# wandb.login()


# api = wandb.Api()


# entity = "loriss"
# project = "linear_coder"


# runs = api.runs(f"{entity}/{project}")


# for run in runs:
#     print(f"Deleting run: {run.name} ({run.id})")
#     run.delete()


In [6]:
# !python3 merge_results.py --source_dir="cache/scoring/partial" --target_dir="results/scoring"

In [7]:
import pandas as pd

df = pd.read_parquet("./results/scoring/merged.parquet")
# assert (df.groupby(["explanation_type", "linear_coder"]).count() == n_test).all().all()


In [8]:
df.columns

Index(['explanation_type', 'model', 'estimator', 'document_idx',
       'train_dataset', 'train_split', 'test_dataset', 'test_split',
       'linear_coder', 'pred_gain', 'mse', 'l1', 'l2', 't'],
      dtype='object')

In [9]:
import re

def extract_k(explanation_type):
    # Case: "The test instance (as a sanity check)"
    if "The test instance (as a sanity check)" in explanation_type:
        return 1

    # Case: "<number> by ... from Top-..."
    first_number_match = re.match(r"^\s*(\d+)\b", explanation_type)
    if first_number_match:
        return int(first_number_match.group(1))

    # Case: "Top-<number> ..." when no leading number
    top_match = re.search(r"Top-(\d+)", explanation_type)
    if top_match:
        return int(top_match.group(1))

    # Case: "<number> random examples"
    random_match = re.search(r"(\d+)\s+random examples", explanation_type)
    if random_match:
        return int(random_match.group(1))

    return None

# Apply to dataframe
df["k"] = df["explanation_type"].apply(extract_k)

# Replace the first occurrence of k in the string with "X"
def replace_k(explanation_type, k):
    if k is None:
        return explanation_type
    # Only replace the first occurrence of the number as a standalone word
    return re.sub(rf"\b{k}\b", "X", explanation_type, count=1)


import numpy as np

def vectorized_replace_k(explanation_types, ks):
    result = explanation_types.copy()
    for k in np.unique(ks[ks.notnull()]):  # only unique, non-null ks
        # Use string pattern, not compiled regex
        pattern = rf"\b{k}\b"
        mask = ks == k
        result.loc[mask] = result.loc[mask].str.replace(pattern, "X", n=1, regex=True)
    return result
df["explanation_type_no_k"] = vectorized_replace_k(df["explanation_type"], df["k"])


def facility_location_hotfix(x):
    if ("facility" in x) and x.startswith("Top-"):
        return x[len("Top-"):]
    else:
        return x
        
df["explanation_type"] = df["explanation_type"].apply(facility_location_hotfix) # hotfix: inconsistent naming scheme for facility location selections


def get_sort_type(x):
    for sort_type in ["scores with largest absolute value", "most positive scores", "most negative scores", "scores closest to zero"]:
        if sort_type in x:
            return sort_type
    return "-"
df["sort_type"] = df["explanation_type"].apply(get_sort_type)

In [10]:
from numpy import trapz
import numpy as np

import re

import os
import torch

In [11]:

def rename_model(x):
       if x == "OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42":
              return "Olmo2-1B"
       
def rename_estimator(x):
      return x.split(":")[0]
def rename_linear_coder(x):
      return x.replace("Coder","").replace("Thresh","")
def extract_seed(x):
       return int(re.search(r'seed (\d+)', x).group(1)) if "seed" in x else None

def rename_random(x):
    return re.sub(r' with seed \d+', "", x)



## Analysis of linear_coders

In [12]:
cols_to_display = ['explanation_type', 'model', 'estimator', "k", 
    #    'train_dataset', 'train_split', 'test_dataset', 'test_split',   'linear_coder',  't', 'k', 'explanation_type_no_k'
     'linear_coder',"l1", "l2", "Prop. non-zero",
       "Prop. negative"
      ]




In [13]:
df["seed"] = df["explanation_type"].apply(extract_seed).fillna(-1)

df["linear_coder"] = df["linear_coder"].apply(rename_linear_coder)
softplus = torch.nn.Softplus()

# df["sum Softplus(t)"] = df["t"].apply(lambda t: float(softplus(torch.tensor(t)).sum()))
# df["sum(t)"] = df["t"].apply(lambda t: sum(t))

df["Prop. non-zero"] = df.apply(lambda row: np.count_nonzero(row["t"]) / len(row["t"]), axis=1)
df["Prop. negative"] = df.apply(lambda row: np.count_nonzero(row["t"] < 0) / len(row["t"]), axis=1)

table = df.groupby(["model", 'train_dataset', 'train_split', 'test_dataset', 'test_split',"estimator", "seed",
       'linear_coder',"explanation_type","explanation_type_no_k","k",]).mean(numeric_only=True).reset_index()[cols_to_display]


table["explanation_type"] = table["explanation_type"].apply(rename_random)

table["model"] = table["model"].apply(rename_model)
table["estimator"] = table["estimator"].apply(rename_estimator)

# macro avg over random selections
table = table.groupby(["k", "linear_coder"]).mean(numeric_only=True).reset_index()[["k", 'linear_coder',
       'l1',"l2","Prop. non-zero", "Prop. negative"
      ]]
# table["k"] = table["k"].apply(lambda x: str(int(x)))
#table = table.set_index(["model", "k"]).sort_values(by=["model", "k",], ascending=False)
# table["mse"] = table["mse"].apply(lambda x: f"{x:.2e}") 

table = table.set_index(["k","linear_coder"]).sort_values(by=["k"])


In [14]:
macro_avg = table.groupby('linear_coder').mean(numeric_only=True).reset_index()
macro_avg['k'] = 'macro_avg' 
table_with_macro = pd.concat([table.reset_index(), macro_avg], ignore_index=True)
table_with_macro = table_with_macro.set_index(['k','linear_coder']).sort_index()


In [15]:
def heatmap_per_group(df, group_col):
    styled = df.style

    for _, group in df.groupby(group_col):
        idx = group.index

        # l1, l2 → lower is better → green for low values
        for col in ['l2', 'l1', "Prop. negative", "Prop. non-zero"]:
            styled = styled.background_gradient(
                cmap='Greens_r', subset=pd.IndexSlice[idx, [col]]
            )

      
       
    return styled
styled_table = heatmap_per_group(table_with_macro.reset_index(), group_col='k')
styled_table

,k,linear_coder,l1,l2,Prop. non-zero,Prop. negative
0,1,KLT,0.000000,0.000000,0.814188,0.381566
1,1,MSE,0.088647,12.340693,0.814188,0.383712
2,1,MSENNLSL2,0.043183,0.031595,0.430255,0.000000
3,1,MSEProjUSimp,1.000000,1.000000,1.000000,0.000000
4,1,MSEProjUSimpSparse,1.000000,1.000000,1.000000,0.000000
5,1,MSEProjUSimpSparseSoft,1.000000,1.000000,1.000000,0.000000
6,5,KLT,0.073362,0.042199,0.921629,0.462931
7,5,MSE,0.075550,0.049114,0.921629,0.459293
8,5,MSENNLSL2,0.141205,4.160811,0.445708,0.000000
9,5,MSEProjUSimp,1.000000,0.209808,0.992200,0.000000


In [16]:
styled_table = heatmap_per_group(macro_avg.reset_index(), group_col='k').hide(axis="columns", subset=["k","index"]).hide(axis="index").format({col: "{:.2f}" for col in styled_table.data.select_dtypes(include="number").columns})
styled_table

linear_coder,l1,l2,Prop. non-zero,Prop. negative
KLT,0.07,0.03,0.90,0.45
MSE,0.09,3.12,0.90,0.45
MSENNLSL2,0.36,3.69,0.43,0.00
MSEProjUSimp,1.00,0.34,0.99,0.00
MSEProjUSimpSparse,1.01,0.47,0.59,0.00
MSEProjUSimpSparseSoft,1.00,0.34,0.99,0.00


In [17]:

# Ensure output folder exists
os.makedirs("./tables", exist_ok=True)

# Export LaTeX with colors
latex_tabular = styled_table.format_index(escape="latex", axis=1).format_index(escape="latex", axis=0).to_latex(
    convert_css=True,   # Converts background colors to \cellcolor
  
    hrules=True,
    column_format='l' + 'c'*(len(styled_table.data.columns))  # Adjust column alignment
)

latex_table = (
    "\\begin{table}[ht]\n"
    "\\scriptsize\n"
    "\\centering\n"
    "\\setlength{\\tabcolsep}{3pt}  % Default is 6pt, reduce for tighter columns\n"
    "\\caption{Statistics for coefficent vector $t$ with different linear coders.}"
    "\\label{tab:linear_coder_selection}\n"
    f"{latex_tabular}\n"
    "\\end{table}\n"
)

with open("./tables/linear_coder_selection.tex", "w") as f:
    f.write(latex_table)

## Quality Score Results per Model and Estimator

In [18]:
df.groupby("estimator").count()

,explanation_type,model,document_idx,train_dataset,train_split,test_dataset,test_split,linear_coder,pred_gain,mse,l1,l2,t,k,explanation_type_no_k,sort_type,seed,Prop. non-zero,Prop. negative
estimator,,,,,,,,,,,,,,,,,,,
"BM25Estimator: k1=1.5, b=0.75",318000,318000,318000,318000,318000,318000,318000,318000,318000,318000,318000,318000,318000,318000,318000,318000,318000,318000,318000
DataInfEstimator: fast_implementation=True,515679,515679,515679,515679,515679,515679,515679,515679,515679,515679,515679,515679,515679,515679,515679,515679,515679,515679,515679
LESSEstimator: normalize=True,510000,510000,510000,510000,510000,510000,510000,510000,510000,510000,510000,510000,510000,510000,510000,510000,510000,510000,510000


In [19]:
cols_to_display = ['explanation_type', 'model', 'estimator', "k","sort_type",#'l1', 'l2',
    #    'train_dataset', 'train_split', 'test_dataset', 'test_split',   'linear_coder',  't', 'k', 'explanation_type_no_k'
       'mse', #'linear_coder', 'pred_gain',
      ]
linear_coder = "MSEProjUSimpSparse"#"MSECoderProjUSimpSparse"




In [72]:
df["seed"] = df["explanation_type"].apply(extract_seed).fillna(-1)
df
df_filtered = df[df["linear_coder"] == linear_coder]
df_filtered = df_filtered[~(df_filtered["explanation_type"].str.contains("facility")) | ((df_filtered["explanation_type"].str.contains("lambda=0.5")))]
df_filtered = df_filtered[
    df_filtered["sort_type"].str.contains("scores with largest absolute value|most negative scores|-", regex=True, case=False)
]
df_at_k_10 = df_filtered[df_filtered["k"] == 10]

In [74]:
for (filename, ddf) in [("full_per_estimator_model_selection_k_10", df_at_k_10), ("full_per_estimator_model_selection", df)]:
       table =ddf.groupby(["model", 'train_dataset', 'train_split', 'test_dataset', 'test_split',"estimator", "seed",
              'linear_coder',"explanation_type","explanation_type_no_k","k","sort_type"]).mean(numeric_only=True).reset_index()[cols_to_display]


       table["explanation_type"] = table["explanation_type"].apply(rename_random)

       table["model"] = table["model"].apply(rename_model)
       table["estimator"] = table["estimator"].apply(rename_estimator)


       # macro avg over random selections
     #  table = table.groupby(['explanation_type', 'model',"sort_type"]).mean(numeric_only=True).reset_index()[cols_to_display]
       
     #  table = table.set_index(["model", "k", "estimator","sort_type"]).sort_values(by=["model", "k","mse"], ascending=True)
       table_agg = table.groupby(['explanation_type', 'model', 'sort_type',]).mean(numeric_only=True).reset_index()
       table_agg['estimator'] = table.groupby(['explanation_type', 'model', 'sort_type'])['estimator'].first().values
       table_agg.loc[table_agg["explanation_type"].str.contains("random"), "estimator"] = "-"
       table = table_agg[cols_to_display]
       table["k"] = table["k"].apply(lambda x: str(int(x)))
       table = table.set_index(["model", "k", "estimator","sort_type"]).sort_values(by=["model", "k","mse"], ascending=True)
       # table["mse"] = table["mse"].apply(lambda x: f"{x:.2e}") 
       # vmin = np.percentile(table.reset_index()["mse"], 5)   # 5th percentile
       # vmax = np.percentile(table.reset_index()["mse"], 95)  # 95th percentile
       
       styled = table.reset_index().style.background_gradient(subset=["mse"], cmap="PuBu", ).format({"mse": "{:.2e}"}).hide(axis="columns", subset=["k","sort_type"]).hide(axis="index")
   
       display(styled)
       os.makedirs("./tables", exist_ok=True)


       latex_tabular = styled.format_index(escape="latex", axis=1).format_index(escape="latex", axis=0).to_latex(
              convert_css=True,   # Converts background colors to \cellcolor
              
              hrules=True,
              column_format='l' + 'l'*(len(styled_table.data.columns))  # Adjust column alignment
              )


       latex_table = (
       "\\begin{table*}[htbp]\n"
       "\\scriptsize\n"
       "\\centering\n"
              f"{latex_tabular}\n"
       "\\end{table*}\n"
       )


       with open(f"./tables/{filename}.tex", "w") as f:
              f.write(latex_table)

model,estimator,explanation_type,mse
Olmo2-1B,-,10 random examples,8.92e-05
Olmo2-1B,BM25Estimator,10 by facility location from Top-100 most influential (scores with largest absolute value). lambda=0.5,3.00e-04
Olmo2-1B,DataInfEstimator,10 by facility location from Top-100 most helpful (most negative scores). lambda=0.5,3.90e-04
Olmo2-1B,BM25Estimator,Top-10 most influential (scores with largest absolute value),4.10e-02
Olmo2-1B,DataInfEstimator,Top-10 most helpful (most negative scores),4.43e-02


model,estimator,explanation_type,mse
Olmo2-1B,DataInfEstimator,1 by facility location from Top-10 least influential (scores closest to zero). lambda=0.0,1.25e-05
Olmo2-1B,BM25Estimator,The test instance (as a sanity check),1.45e-05
Olmo2-1B,DataInfEstimator,1 by facility location from Top-100 least influential (scores closest to zero). lambda=0.75,3.05e-05
Olmo2-1B,DataInfEstimator,1 by facility location from Top-100 least influential (scores closest to zero). lambda=0.25,5.02e-05
Olmo2-1B,-,1 random examples,8.98e-05
Olmo2-1B,BM25Estimator,1 by facility location from Top-100 least influential (scores closest to zero). lambda=0.0,9.14e-05
Olmo2-1B,BM25Estimator,1 by facility location from Top-100 least influential (scores closest to zero). lambda=0.5,9.18e-05
Olmo2-1B,BM25Estimator,1 by facility location from Top-100 least influential (scores closest to zero). lambda=1.0,9.20e-05
Olmo2-1B,BM25Estimator,Top-1 least influential (scores closest to zero),1.36e-04
Olmo2-1B,BM25Estimator,1 by facility location from Top-100 most influential (scores with largest absolute value). lambda=0.0,1.43e-04


In [29]:
df.linear_coder.unique()

array(['MSEProjUSimp', 'KLT', 'MSE', 'MSENNLSL2', 'MSEProjUSimpSparse',
       'MSEProjUSimpSparseSoft'], dtype=object)

In [39]:
results = []
for name, group in df.groupby(["model", 'train_dataset', 'train_split', 'test_dataset', 'test_split',"estimator",
       'linear_coder',"explanation_type_no_k","k",]).mean(numeric_only=True).groupby(["model", 'train_dataset', 'train_split', 'test_dataset', 'test_split',"estimator",
       'linear_coder',"explanation_type_no_k"]):
              
       dff = group.reset_index().sort_values(by="k")
       dff = dff[dff["linear_coder"] == "MSEProjUSimpSparse"]
       # display(dff)
       # break
       if len(dff) < 2:
              continue
       # display(group)
       # break
       
       
   
       result_dict = dff.iloc[0][[
              'model', 'train_dataset', 'train_split', 'test_dataset', 'test_split',"estimator",
              'linear_coder', 'explanation_type_no_k'
       ]].to_dict()

       result_dict['auc_pred_gain'] = np.trapz(dff["pred_gain"], dff["k"])
       result_dict['mean_pred_gain'] = dff["pred_gain"].mean()
       result_dict['auc_mse'] = np.trapz(dff["mse"], dff["k"])
       result_dict['mean_mse'] = dff["mse"].mean()
       result_dict['count'] = len(dff)
       results.append(result_dict)
table = pd.DataFrame(results).sort_values(by="mean_mse")[["model", "estimator", "explanation_type_no_k", "mean_mse", "auc_mse", ]]



table["model"] = table["model"].apply(rename_model)

table["estimator"] = table["estimator"].apply(rename_estimator)




In [42]:
os.makedirs("./tables", exist_ok=True)


latex_tabular = table.to_latex(
    index=False,
    longtable=False,
    float_format="%.4f",
    escape=True,
)


latex_table = (
    "\\begin{table*}[htbp]\n"
    "\\scriptsize\n"
    "\\centering\n"
    "\\caption{Aggregate results across budgets for different selection strategies "
    "(mean or auc across $k=\\{1,5,10,25\\}$)}\n"
    "\\label{tab:aggregate_results}\n"
    f"{latex_tabular}\n"
    "\\end{table*}\n"
)


with open("./tables/aggregate_results.tex", "w") as f:
    f.write(latex_table)

In [ ]:
import re

def get_tuples():
    results = []
    explanation_types = df["explanation_type"].unique()
    top_items = {re.match(r'Top-(\d+)\s+(.*)', d).groups()
                 for d in explanation_types if d.startswith("Top-")}
    
    for d in explanation_types:
        if "by facility location" not in d:
            continue
        
  
        num_match = re.match(r'(\d+)\s+by facility location.*from Top-\d+\s+(.*)\. lambda', d)
        if not num_match:
            continue
        
        num, typ = num_match.groups()
        typ = typ.strip()  
        
       
        match = next((f"Top-{n} {t}" for n, t in top_items if n == num and t == typ), None)
        if match:
            assert d in explanation_types and match in explanation_types
            results.append((d, match))
    
    return results

df_comp = []
for fl_selection_type, naive_selection_type in get_tuples():
    df_filtered = df[df["explanation_type"].isin([fl_selection_type, naive_selection_type])].copy()
    df_filtered["type"] = df_filtered["explanation_type"].map({
        fl_selection_type: "fl",
        naive_selection_type: "naive"
    })

    df_filtered = df_filtered#[df_filtered["linear_coder"] == "MSECoderProjUSimp"]
    df_m = (
    
        df_filtered.groupby(
            [
                "model", "estimator", #"document_idx",
                "train_dataset", "train_split", "test_dataset",
                "test_split", "linear_coder", "type"
            ]
        ).mean(numeric_only=True)
    ).reset_index()
    df_m["setting"] = fl_selection_type
    df_comp.append(df_m.pivot_table(index=["model", "estimator", "train_dataset", "train_split",
                        "test_dataset", "test_split", "linear_coder","setting"],
                 columns="type",
                 values=["pred_gain", "mse"]).reset_index())#, "l1", "l2"]))
    
df_comp = pd.concat(df_comp).dropna()


In [ ]:
df_comp

In [ ]:
df_comp

In [ ]:
df_comp

In [ ]:
df_comp["mse_reduction_pct"] = (1 - df_comp[('mse', 'fl')] / df_comp[('mse', 'naive')]) * 100


table = df_comp.sort_values(ascending=False,by=('mse_reduction_pct'))[["model", "estimator",  "setting", "mse_reduction_pct"]]



table["model"] = table["model"].apply(rename_model)

table["estimator"] = table["estimator"].apply(rename_estimator)
os.makedirs("./tables", exist_ok=True)


latex_tabular = table.to_latex(
    index=False,
    longtable=False,
    float_format="%.4f",
    escape=True,
)


latex_table = (
    "\\begin{table*}[htbp]\n"
    "\\scriptsize\n"
    "\\centering\n"
    "\\caption{Reduction in mse in pct of facility location based strategy vs. naiive.}\n"
    "\\label{tab:mse_reduction}\n"
    f"{latex_tabular}\n"
    "\\end{table*}\n"
)


with open("./tables/mse_reduction.tex", "w") as f:
    f.write(latex_table)

In [ ]:
table

In [ ]:
dfas

In [ ]:
# %run score.py --explanation_type=FacilityLocationMostHarmful

In [ ]:
df_comp["pct_increase_mse"] = (df_comp[('mse','fl')] / df_comp[('mse','naive')] - 1) * 100
df_comp["pct_increase_pred_gain"] = (df_comp[('pred_gain','fl')] / df_comp[('pred_gain','naive')] - 1) * 100
df_comp

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Flatten MultiIndex
df_flat = df_comp.copy()
df_flat.columns = ['_'.join(filter(None, col)) for col in df_flat.columns.values]

# Reshape to long format
df_long = pd.wide_to_long(
    df_flat,
    stubnames=['mse', 'pred_gain'],
    i='setting',
    j='method',
    sep='_',
    suffix='.*'
).reset_index()

# Melt metrics to have 'metric' and 'value'
df_long2 = pd.melt(
    df_long,
    id_vars=['setting', 'method'],
    value_vars=['mse', 'pred_gain'],
    var_name='metric',
    value_name='value'
)

g = sns.FacetGrid(
    df_long2,
    row='metric',
    col='method',
    sharey=True,
    height=4,
    aspect=1.5,
    hue='setting',   # <-- add hue
    palette="Set2"
)

g.map_dataframe(sns.barplot, x='setting', y='value', order=df_flat['setting'].unique())

# Add legend once for all facets
g.add_legend(title="Setting")

# Rotate x-axis labels
for ax in g.axes.flatten():
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Custom facet titles
g.set_titles(row_template="{row_name}", col_template="{col_name}")

g.fig.subplots_adjust(bottom=0.25, hspace=0.3)
plt.show()



In [ ]:
ogmflörfv

In [ ]:
# df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('l1', 'mean')].sort_values()

In [ ]:
mean_mse = df.groupby(["explanation_type", "linear_coder", "estimator"])['mse'].mean().reset_index()
mean_mse['rank'] = mean_mse.groupby(['explanation_type', 'linear_coder'])['mse'].rank(method='min')
mean_mse = mean_mse.sort_values(['linear_coder', 'explanation_type', 'rank'])
mean_mse

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_palette("Set2")

mean_mse = df.groupby(["explanation_type", "linear_coder", "estimator"])['mse'].mean().reset_index()
mean_mse = mean_mse[~mean_mse["explanation_type"].str.contains("random")]



best_estimators = mean_mse.loc[mean_mse.groupby(['explanation_type', 'linear_coder'])['mse'].idxmin()]
plt.tight_layout()

estimators = best_estimators['estimator'].unique()
markers = ['o', 's', 'D', '^', 'v', '<', '>', 'P', 'X', '*']  
colors = sns.color_palette("Set2", n_colors=len(estimators)) 
marker_map = {est: markers[i % len(markers)] for i, est in enumerate(estimators)}
color_map = {est: colors[i % len(colors)] for i, est in enumerate(estimators)}

explanation_types = list(best_estimators['explanation_type'].unique())
linear_coders = list(best_estimators['linear_coder'].unique())

fig, ax = plt.subplots(figsize=(6,6))

for _, row in best_estimators.iterrows():
    x = linear_coders.index(row['linear_coder'])  
    y = explanation_types.index(row['explanation_type'])  
    ax.scatter(
        x, y,
        marker=marker_map[row['estimator']],
        color=color_map[row['estimator']],
        s=200
    )

ax.set_xticks(range(len(linear_coders)))
ax.set_xticklabels(linear_coders)
ax.set_yticks(range(len(explanation_types)))
ax.set_yticklabels(explanation_types)
ax.set_xlabel('Linear Coder') 
ax.set_ylabel('Explanation Type')  
ax.set_xticklabels(linear_coders, rotation=45, ha='right')  

for est in estimators:
    ax.scatter([], [], marker=marker_map[est], color=color_map[est], label=est, s=200)
ax.legend(title='Best Estimator', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()


In [ ]:
df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('mse', 'mean')].sort_values()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
plt.tight_layout()

filtered_df = df.copy()  

estimators = filtered_df['estimator'].unique()
if len(estimators) != 2:
    raise ValueError("This plot assumes exactly 2 estimators.")

max_per_group = filtered_df.groupby('linear_coder')['pred_gain'].max().sort_values(ascending=False)
sorted_linear_coders = max_per_group.index.tolist()

g = sns.catplot(
    data=filtered_df,
    x="explanation_type",
    y="pred_gain",
    hue="estimator",
    col="linear_coder",
    col_order=sorted_linear_coders,
    kind="box",
    palette="Set2",
    dodge=True,



)

for ax in g.axes.flat:
    ax.axhline(1.0, color='black', linestyle='dashed')



g.set_xticklabels(rotation=90, ha='right')


g.fig.subplots_adjust(wspace=0.0) 


plt.show()
